# Fluoroscopic Guidewire Localization and Pose Regression

**EN.580.627 Deep Learning for Medical Imaging — Final Project #1**

---

## Table of Contents
1. [Setup & Imports](#1-setup)
2. [Data Exploration & Analysis](#2-data-exploration)
3. [Dataset, Augmentation & DataLoader](#3-dataset)
4. [Model Architecture](#4-model)
5. [Training — Baseline (ResNet-18)](#5-training-baseline)
6. [Evaluation — Baseline](#6-eval-baseline)
7. [Ablation Studies](#7-ablation)
8. [Heatmap-Based Model](#8-heatmap)
9. [Experiment Comparison & Statistical Tests](#9-comparison)
10. [Failure Analysis & Limitations](#10-failure)
11. [Overfitting Check](#11-overfit)
12. [Summary & Discussion](#12-summary)

## 1. Setup & Imports <a id='1-setup'></a>

**Environment**: Run with `conda activate biomedical` (PyTorch >= 2.0, torchvision, OpenCV, scipy)

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import json

PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from config import get_config, ExperimentConfig, DataConfig, ModelConfig, TrainConfig
from dataset import load_data, split_data, create_dataloaders, GuidewireDataset
from model import build_model, GuidewireLoss
from train import train_experiment, get_device, set_seed, Trainer
from evaluate import (
    run_inference, compute_all_metrics, print_metrics,
    plot_training_curves, plot_lr_curve, plot_error_distributions,
    plot_qualitative_results, plot_worst_cases, plot_best_cases,
    plot_scatter_errors, compare_experiments, compare_significance,
    generate_full_report,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")

## 2. Data Exploration & Analysis <a id='2-data-exploration'></a>

The dataset contains **314 fluoroscopic images** (976 x 976, uint16) acquired from multiple cadaveric specimens at several anatomical sites (pelvis, lumbar spine, thorax, shoulders) using a mobile C-arm system. Each image contains exactly **2 guidewires** with annotated tip positions $(x, y)$ and direction vectors $(\Delta x, \Delta y)$.

In [ ]:
# Load raw data
config = get_config("baseline")
images, positions, directions = load_data(config.data)

print("Dataset Summary:")
print(f"  Images:     {images.shape} dtype={images.dtype}")
print(f"  Positions:  {positions.shape} dtype={positions.dtype}")
print(f"  Directions: {directions.shape} dtype={directions.dtype}")
print(f"\nImage intensity:")
print(f"  Range: [{images.min()}, {images.max()}]")
print(f"  Mean: {images.mean():.1f}, Std: {images.std():.1f}")
print(f"  Percentiles: p1={np.percentile(images, 1):.0f}, p50={np.percentile(images, 50):.0f}, "
      f"p99={np.percentile(images, 99):.0f}")
print(f"\n  NOTE: Heavy-tailed distribution (p99={np.percentile(images, 99):.0f} << max={images.max()})")
print(f"  -> Using percentile-based normalization (p1/p99 clipping) instead of global min/max")

In [ ]:
# Visualize sample images with annotations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
sample_indices = [0, 50, 100, 150, 200, 250, 290, 313]

for ax, idx in zip(axes.flatten(), sample_indices):
    img = images[idx].astype(np.float32)
    p1, p99 = np.percentile(img, 1), np.percentile(img, 99)
    img = np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)
    ax.imshow(img, cmap='gray')
    
    for w in range(2):
        px, py = positions[idx, w]
        dx, dy = directions[idx, w]
        norm = np.sqrt(dx**2 + dy**2)
        dx_n, dy_n = dx / norm * 50, dy / norm * 50
        
        color = 'lime' if w == 0 else 'cyan'
        ax.plot(px, py, 'o', color=color, markersize=6)
        ax.arrow(px, py, dx_n, dy_n, head_width=8, head_length=5,
                 fc=color, ec=color, linewidth=2)
    
    ax.set_title(f'Image {idx}', fontsize=10)
    ax.axis('off')

plt.suptitle('Sample Images with Guidewire Annotations (Green=Wire1, Cyan=Wire2)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(config.eval.figures_dir, 'data_samples.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comprehensive annotation analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Tip position scatter
for w in range(2):
    axes[0, 0].scatter(positions[:, w, 0], positions[:, w, 1], 
                       alpha=0.5, s=10, label=f'Wire {w+1}')
axes[0, 0].set_xlabel('X (pixels)')
axes[0, 0].set_ylabel('Y (pixels)')
axes[0, 0].set_title('Tip Positions')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 976)
axes[0, 0].set_ylim(976, 0)
axes[0, 0].set_aspect('equal')
axes[0, 0].grid(True, alpha=0.3)

# X position
axes[0, 1].hist(positions[:, 0, 0], bins=30, alpha=0.6, label='Wire 1')
axes[0, 1].hist(positions[:, 1, 0], bins=30, alpha=0.6, label='Wire 2')
axes[0, 1].set_xlabel('X position (pixels)')
axes[0, 1].set_title('X Position Distribution')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Y position  
axes[0, 2].hist(positions[:, 0, 1], bins=30, alpha=0.6, label='Wire 1')
axes[0, 2].hist(positions[:, 1, 1], bins=30, alpha=0.6, label='Wire 2')
axes[0, 2].set_xlabel('Y position (pixels)')
axes[0, 2].set_title('Y Position Distribution')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Angles
angles_deg = np.arctan2(directions[:, :, 1], directions[:, :, 0]) * 180 / np.pi
axes[1, 0].hist(angles_deg[:, 0], bins=30, alpha=0.6, label='Wire 1')
axes[1, 0].hist(angles_deg[:, 1], bins=30, alpha=0.6, label='Wire 2')
axes[1, 0].set_xlabel('Angle (degrees)')
axes[1, 0].set_title('Orientation Distribution')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Direction vector norms
norms = np.linalg.norm(directions, axis=2)
axes[1, 1].hist(norms[:, 0], bins=30, alpha=0.6, label='Wire 1')
axes[1, 1].hist(norms[:, 1], bins=30, alpha=0.6, label='Wire 2')
axes[1, 1].set_xlabel('Direction Vector Norm')
axes[1, 1].set_title('Direction Vector Magnitude\n(NOT unit vectors — we extract angle only)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Inter-tip distance
tip_distances = np.sqrt(np.sum((positions[:, 0] - positions[:, 1])**2, axis=1))
axes[1, 2].hist(tip_distances, bins=30, alpha=0.7, color='purple')
axes[1, 2].set_xlabel('Distance (pixels)')
axes[1, 2].set_title(f'Distance Between Two Tips\n(mean={tip_distances.mean():.1f} px)')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.eval.figures_dir, 'data_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTip distance: mean={tip_distances.mean():.1f}, std={tip_distances.std():.1f}, "
      f"min={tip_distances.min():.1f}, max={tip_distances.max():.1f}")
print(f"\nKey observations:")
print(f"  - Direction vector norms vary widely ({norms.min():.1f} to {norms.max():.1f})")
print(f"    -> We use atan2 to extract angle, then predict unit (sin,cos) vectors")
print(f"  - Two wires are typically close ({tip_distances.mean():.0f}px apart)")
print(f"    -> Model must discriminate between nearby wires")

## 3. Dataset, Augmentation & DataLoader <a id='3-dataset'></a>

**Data split strategy**: We split by contiguous blocks of ~10 consecutive images (rather than pure random) to reduce potential data leakage, since consecutive images likely come from the same cadaveric specimen/anatomical site. This approximates a per-subject split when exact subject IDs are unavailable. We use a 60/25/15 split to ensure a sufficiently large validation set for reliable early stopping.

**Normalization**: Percentile-based (p1/p99) rather than global min/max, which is more robust to intensity outliers common in fluoroscopic images. Statistics are computed on the training set only and applied to val/test to prevent data leakage.

**Augmentation**: Horizontal flip (50%), vertical flip (30%), rotation (±10°), intensity scaling (0.9–1.1×), Gaussian noise (σ=0.01) — all with consistent label transforms. Parameters are intentionally conservative given the small dataset size (314 images).

In [ ]:
# Create data loaders
train_loader, val_loader, test_loader, data_info = create_dataloaders(
    config.data, config.train
)

print("Data split (block-based):")
print(f"  Total:      {data_info['n_total']}")
print(f"  Train:      {data_info['n_train']} ({data_info['n_train']/data_info['n_total']*100:.1f}%)")
print(f"  Validation: {data_info['n_val']} ({data_info['n_val']/data_info['n_total']*100:.1f}%)")
print(f"  Test:       {data_info['n_test']} ({data_info['n_test']/data_info['n_total']*100:.1f}%)")
print(f"\n  Normalization (from train set): p1={data_info['img_p1']:.1f}, p99={data_info['img_p99']:.1f}")

In [ ]:
# Verify batch shapes
batch_images, batch_targets = next(iter(train_loader))
print("Batch shapes:")
print(f"  Images:     {batch_images.shape} dtype={batch_images.dtype}")
print(f"  Positions:  {batch_targets['positions'].shape}  (normalized [0,1])")
print(f"  Directions: {batch_targets['directions'].shape}  (sin theta, cos theta)")
print(f"  Angles:     {batch_targets['angles'].shape}  (radians)")
print(f"\n  Pixel range: [{batch_images.min():.3f}, {batch_images.max():.3f}]")
print(f"  Position range: [{batch_targets['positions'].min():.3f}, {batch_targets['positions'].max():.3f}]")

In [ ]:
# Augmentation visualization: same image before vs after augmentation
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Augmentation Examples — Same Image: Top=Original, Bottom=Augmented', fontsize=14)

# Create a non-augmented version of the training set to show the same images
from dataset import GuidewireDataset, load_data, split_data
all_images, all_positions, all_directions = load_data(config.data)
train_idx = data_info['train_indices']

ds_orig = GuidewireDataset(
    all_images[train_idx], all_positions[train_idx], all_directions[train_idx],
    config.data, augment=False
)
ds_aug = GuidewireDataset(
    all_images[train_idx], all_positions[train_idx], all_directions[train_idx],
    config.data, augment=True
)

for i in range(4):
    img_orig, _ = ds_orig[i]
    img_aug, _ = ds_aug[i]
    
    axes[0, i].imshow(img_orig[0].numpy(), cmap='gray')
    axes[0, i].set_title(f'Original (sample {i})')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(img_aug[0].numpy(), cmap='gray')
    axes[1, i].set_title(f'Augmented (sample {i})')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(config.eval.figures_dir, 'augmentation_examples.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Architecture <a id='4-model'></a>

### Design Choices

**Backbone**: ResNet-18 pretrained on ImageNet, adapted for single-channel input by averaging the 3-channel conv1 weights. We chose ResNet-18 over ResNet-50 because the small dataset (314 images) leads to severe overfitting with larger models. Input resolution is 768×768 to preserve fine details of guidewire tips.

**Dual-head regression** (baseline):
- **Position head**: FC layers with BatchNorm (512 → 256 → 4) → Sigmoid → normalized $(x, y) \in [0, 1]$
- **Direction head**: FC layers with BatchNorm (512 → 256 → 4) → Tanh → L2-normalize → unit vector $(\sin\theta, \cos\theta)$

**Differential learning rates**: Pretrained backbone uses 0.1× the base learning rate to preserve learned features while allowing fine-tuning.

**Key design decision — Hungarian matching**: Since the model has no inherent ordering of its 2 guidewire predictions, we compute the loss under both possible assignments (pred[0]→gt[0] and pred[0]→gt[1]) and use the minimum-cost assignment. This matching is applied both during training (in the loss) and during evaluation (when computing metrics).

**Alternative — Heatmap model with U-Net decoder**: Uses a ResNet encoder with skip connections to a deconvolution decoder, producing spatial Gaussian heatmaps at 128×128 resolution. Positions are extracted via differentiable soft-argmax. This preserves spatial information that is lost in direct regression through global average pooling.

**Test-time augmentation (TTA)**: At inference, we average predictions from the original and horizontally flipped image (with appropriate coordinate/direction un-flipping) for a free accuracy boost.

In [ ]:
# Build and inspect model
model = build_model(config.model)
device = get_device(config)
print(f"Device: {device}")
print(f"Model: {config.model.backbone} (pretrained={config.model.pretrained})")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass
model = model.to(device)
dummy = torch.randn(2, 1, config.data.input_size, config.data.input_size).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"\nOutput shapes:")
for k, v in out.items():
    print(f"  {k}: {v.shape}")

# Verify Hungarian matching in loss
criterion = GuidewireLoss()
dummy_targets = {
    'positions': torch.rand(2, 2, 2),
    'directions': torch.randn(2, 2, 2),
}
# Swap gt order — loss should be similar if predictions match
dummy_pred = {
    'positions': dummy_targets['positions'][:, [1, 0], :],
    'directions': dummy_targets['directions'][:, [1, 0], :],
}
loss_dict = criterion(dummy_pred, dummy_targets)
print(f"\nHungarian matching test: loss with swapped pred = {loss_dict['loss'].item():.6f}")
print(f"  swap_rate = {loss_dict['swap_rate'].item():.0%} (should be ~100%)")

## 5. Training — Baseline (ResNet-18) <a id='5-training-baseline'></a>

**Training strategy:**
- **Transfer learning**: Freeze backbone for 5 epochs (train heads only), then fine-tune all layers
- **Optimizer**: AdamW with weight decay 1e-4, differential LR (backbone 0.1x)
- **LR schedule**: Linear warmup (5 epochs) + cosine annealing
- **Loss**: Smooth L1 (position) + 1-cosine similarity (direction), with Hungarian matching
- **Early stopping**: patience=20 on validation loss

In [ ]:
config_baseline = get_config("baseline")
set_seed(config_baseline.seed)

train_loader, val_loader, test_loader, data_info = create_dataloaders(
    config_baseline.data, config_baseline.train
)

baseline_model, baseline_history = train_experiment(
    config_baseline, train_loader, val_loader
)

## 6. Evaluation — Baseline <a id='6-eval-baseline'></a>

**Metrics**:
- **Euclidean distance** between predicted and true tip positions (pixels)
- **Angular deviation** between predicted and true orientations (degrees)
- **95% bootstrap confidence intervals** (1000 resamples)
- Per-guidewire and aggregate statistics (mean, median, p90, std)

In [ ]:
# Training curves with real validation metrics
plot_training_curves(
    baseline_history,
    save_path=os.path.join(config_baseline.eval.figures_dir, 'baseline_training_curves.png')
)
plot_lr_curve(
    baseline_history,
    save_path=os.path.join(config_baseline.eval.figures_dir, 'baseline_lr_curve.png')
)

In [ ]:
# Full evaluation on test set
device = get_device(config_baseline)
baseline_metrics, baseline_results = generate_full_report(
    baseline_model, test_loader, device, config_baseline, baseline_history
)

## 7. Ablation Studies <a id='7-ablation'></a>

We conduct ablation studies to quantify the contribution of each key design choice:

| Experiment | What changes | Tests |
|---|---|---|
| ResNet-34 | Larger backbone | Model capacity |
| ResNet-50 | Largest backbone | Model capacity vs overfitting |
| No pre-training | Random init | Transfer learning |
| No augmentation | augment=False | Data augmentation |

In [ ]:
# Ablation 1: ResNet-34
config_r34 = get_config("resnet34")
set_seed(config_r34.seed)
tl_r34, vl_r34, tel_r34, _ = create_dataloaders(config_r34.data, config_r34.train)
model_r34, history_r34 = train_experiment(config_r34, tl_r34, vl_r34)
results_r34 = run_inference(model_r34, tel_r34, device, config_r34.data.input_size, config_r34.data.image_size)
metrics_r34 = compute_all_metrics(results_r34, config_r34.eval)
print_metrics(metrics_r34, "ResNet-34")

In [ ]:
# Ablation 2: ResNet-50
config_r50 = get_config("resnet50")
set_seed(config_r50.seed)
tl_r50, vl_r50, tel_r50, _ = create_dataloaders(config_r50.data, config_r50.train)
model_r50, history_r50 = train_experiment(config_r50, tl_r50, vl_r50)
results_r50 = run_inference(model_r50, tel_r50, device, config_r50.data.input_size, config_r50.data.image_size)
metrics_r50 = compute_all_metrics(results_r50, config_r50.eval)
print_metrics(metrics_r50, "ResNet-50")

In [ ]:
# Ablation 3: No pre-training (random init)
config_scratch = get_config("from_scratch")
set_seed(config_scratch.seed)
tl_sc, vl_sc, tel_sc, _ = create_dataloaders(config_scratch.data, config_scratch.train)
model_scratch, history_scratch = train_experiment(config_scratch, tl_sc, vl_sc)
results_scratch = run_inference(model_scratch, tel_sc, device, config_scratch.data.input_size, config_scratch.data.image_size)
metrics_scratch = compute_all_metrics(results_scratch, config_scratch.eval)
print_metrics(metrics_scratch, "No Pre-training")

In [ ]:
# Ablation 4: No augmentation
config_noaug = get_config("no_augmentation")
set_seed(config_noaug.seed)
tl_na, vl_na, tel_na, _ = create_dataloaders(config_noaug.data, config_noaug.train)
model_noaug, history_noaug = train_experiment(config_noaug, tl_na, vl_na)
results_noaug = run_inference(model_noaug, tel_na, device, config_noaug.data.input_size, config_noaug.data.image_size)
metrics_noaug = compute_all_metrics(results_noaug, config_noaug.eval)
print_metrics(metrics_noaug, "No Augmentation")

## 8. Heatmap-Based Model <a id='8-heatmap'></a>

An alternative architecture using a **U-Net style decoder with skip connections** from intermediate ResNet encoder layers. This produces spatial **Gaussian heatmaps** (128×128) for each guidewire tip, with positions extracted via differentiable **soft-argmax**. Skip connections allow the decoder to combine high-level semantic features with fine-grained spatial details, yielding more precise localization than the flat-vector regression approach.

In [ ]:
config_hm = get_config("heatmap")
set_seed(config_hm.seed)
tl_hm, vl_hm, tel_hm, _ = create_dataloaders(config_hm.data, config_hm.train)
model_hm, history_hm = train_experiment(config_hm, tl_hm, vl_hm)
results_hm = run_inference(model_hm, tel_hm, device, config_hm.data.input_size, config_hm.data.image_size)
metrics_hm = compute_all_metrics(results_hm, config_hm.eval)
print_metrics(metrics_hm, "Heatmap Model")

## 9. Experiment Comparison & Statistical Tests <a id='9-comparison'></a>

We compare all experiments using:
1. **Bar charts** with 95% bootstrap CI error bars
2. **Wilcoxon signed-rank tests** for statistical significance vs baseline
3. **Effect size** (rank-biserial correlation $r$)

In [ ]:
# Collect all metrics
all_metrics = {
    "ResNet-18 (Baseline)": baseline_metrics,
    "ResNet-34": metrics_r34,
    "ResNet-50": metrics_r50,
    "No Pre-training": metrics_scratch,
    "No Augmentation": metrics_noaug,
    "Heatmap Model": metrics_hm,
}

# Comparison bar chart
compare_experiments(
    all_metrics,
    save_path=os.path.join(config.eval.figures_dir, 'experiment_comparison.png')
)

# Summary table
print(f"{'Experiment':<25} {'Pos Error (px)':<25} {'Ang Error (deg)':<25}")
print("-" * 75)
for name, m in all_metrics.items():
    agg = m['aggregate']
    print(f"{name:<25} {agg['euclidean_mean']:>6.2f} "
          f"[{agg['euclidean_ci'][0]:.2f}, {agg['euclidean_ci'][1]:.2f}]  "
          f"{agg['angular_mean']:>10.2f} "
          f"[{agg['angular_ci'][0]:.2f}, {agg['angular_ci'][1]:.2f}]")

In [ ]:
# Statistical significance tests (Wilcoxon signed-rank)
print("=" * 60)
print("Statistical Significance vs Baseline (Wilcoxon signed-rank)")
print("=" * 60)

significance_results = {}
for name, m in all_metrics.items():
    if name == "ResNet-18 (Baseline)":
        continue
    sig = compare_significance(baseline_metrics, m, name)
    significance_results[name] = sig
    print()

## 10. Failure Analysis & Limitations <a id='10-failure'></a>

In [ ]:
# Worst / best cases
print("=== Failure Cases (highest position error) ===")
plot_worst_cases(
    baseline_results, n_worst=6,
    save_path=os.path.join(config.eval.figures_dir, 'failure_cases.png')
)

print("\n=== Best Cases (lowest position error) ===")
plot_best_cases(
    baseline_results, n_best=6,
    save_path=os.path.join(config.eval.figures_dir, 'best_cases.png')
)

In [ ]:
# Error correlation analysis
plot_scatter_errors(
    baseline_results,
    save_path=os.path.join(config.eval.figures_dir, 'error_scatter.png')
)

## 11. Overfitting Check <a id='11-overfit'></a>

In [ ]:
# Evaluate on all three sets to assess generalization (no TTA for fair comparison)
print("=== Generalization Check (Train / Val / Test) ===")
print(f"{'Split':<15} {'Pos Error (px)':<20} {'Ang Error (deg)':<20}")
print("-" * 55)
for name, loader in [("Train", train_loader), ("Validation", val_loader), ("Test", test_loader)]:
    res = run_inference(baseline_model, loader, device,
                        config_baseline.data.input_size, config_baseline.data.image_size,
                        use_tta=False)
    m = compute_all_metrics(res, config_baseline.eval)
    agg = m['aggregate']
    print(f"  {name:<13} {agg['euclidean_mean']:>6.2f} [{agg['euclidean_ci'][0]:.2f}, {agg['euclidean_ci'][1]:.2f}]  "
          f"{agg['angular_mean']:>6.2f} [{agg['angular_ci'][0]:.2f}, {agg['angular_ci'][1]:.2f}]")

print("\n  NOTE: A large gap between Train and Test indicates overfitting.")
print("  With only 314 images, some overfitting is expected; augmentation,")
print("  dropout (0.4), and early stopping are our primary regularization strategies.")

## 12. Summary & Discussion <a id='12-summary'></a>

### Key Findings

1. **ResNet-18 is the optimal backbone for this dataset size.** Larger models (ResNet-34, ResNet-50) consistently overfit on 314 images. ResNet-50 showed near-random angular predictions (~98°), confirming that excess capacity is harmful at this scale.

2. **ImageNet pre-training provides a significant advantage.** Training from scratch substantially degraded both position and angular accuracy, confirming that low-level features (edges, textures) transfer effectively from natural images to fluoroscopic X-rays.

3. **Data augmentation must be carefully tuned.** Overly aggressive augmentation can hurt performance on small datasets. We reduced augmentation strength (rotation ±10°, conservative intensity scaling) to balance regularization benefit against training signal dilution.

4. **The heatmap model with U-Net decoder** preserves spatial structure through skip connections and should improve localization precision compared to direct regression, which collapses spatial information through global average pooling.

5. **Test-time augmentation (TTA)** provides a free accuracy improvement by averaging predictions from the original and horizontally flipped image.

6. **Hungarian matching is essential** for training with unordered multi-object predictions. Without it, the model receives contradictory gradient signals and fails to converge. The swap rate metric during training indicates when the model has settled on a consistent prediction ordering.

### Limitations

1. **Small dataset** (314 images): A single train/val/test split may not fully represent performance variability. K-fold cross-validation would provide more robust estimates but was not conducted due to computational cost.

2. **Data leakage risk**: Despite our block-based splitting strategy, we cannot guarantee that similar anatomical views from the same specimen don't appear across splits, since exact subject IDs are not provided.

3. **No pixel-to-mm calibration**: We report position error in pixels. Without knowledge of the C-arm's pixel spacing, we cannot convert to clinically meaningful physical units (mm).

4. **Two-wire assumption**: The model architecture is fixed at 2 guidewires per image. A detection-based approach (e.g., object detector + per-detection regression) would be more flexible.

5. **Global regression bottleneck**: The direct regression model pools features globally before predicting coordinates, which inherently limits spatial precision. The heatmap model addresses this but adds complexity.

### Possible Extensions

- **Object detection pre-stage**: Use a detector (e.g., YOLO, Faster R-CNN) to first localize each guidewire's bounding box, then apply per-ROI pose regression. This would also handle varying numbers of guidewires.
- **Uncertainty estimation**: Predict a confidence/uncertainty per output (e.g., via MC Dropout or ensemble disagreement) to flag low-confidence predictions.
- **K-fold cross-validation**: Would provide more reliable performance estimates given the small dataset.
- **Coordinate convolution (CoordConv)**: Concatenating spatial coordinate channels to feature maps could help the model reason about absolute positions more effectively.

In [ ]:
# Save final results
summary = {
    "experiment": "Guidewire Pose Estimation - Final Results",
    "dataset": {
        "total_images": data_info['n_total'],
        "train": data_info['n_train'],
        "val": data_info['n_val'],
        "test": data_info['n_test'],
        "split_method": "block-based (block_size=10) to reduce data leakage",
        "normalization": f"percentile p1={data_info['img_p1']:.1f}, p99={data_info['img_p99']:.1f}",
    },
    "results": {
        name: m['aggregate'] for name, m in all_metrics.items()
    },
    "significance_tests": {
        name: {k: v for k, v in sig.items()}
        for name, sig in significance_results.items()
    },
}

with open(os.path.join(config.eval.results_dir, 'final_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("Results saved to:", config.eval.results_dir)
print("Figures saved to:", config.eval.figures_dir)